# CUDA COLMAP feasibility -- headless GPU SIFT on Colab

**One question:** can we get a CUDA-enabled COLMAP working headless on a Colab GPU
runtime, and how fast is it vs CPU?

Today `bootstrap.sh` installs COLMAP via `apt`. That build has **no CUDA**: its OpenGL
SiftGPU backend crashes on headless VMs (no display), so every pipeline run passes
`--colmap-cpu` and burns 1-3 h of A100 credits on exhaustive CPU matching. GPU SIFT is
an order of magnitude faster. If this notebook passes, a follow-up wires
`--colmap-cuda` into `bootstrap.sh`.

**Two rungs, tried in order:**
1. **Rung 1** -- conda-forge binary (~5 min, no compile)
2. **Rung 2** -- source build from GitHub (~20-35 min, fallback if rung 1 fails)

**Standalone:** this notebook does NOT clone the project repo. It is a pure build
experiment. Runtime: **A100 GPU** (Runtime -> Change runtime type -> GPU).

## 0. GPU + environment check

In [ ]:
import os, subprocess, time, json

# Global state -- init here so every later cell can reference safely, even on re-run.
RUNG_USED    = 0          # 1 = conda-forge binary, 2 = source build, 0 = nothing yet
COLMAP_BIN   = ''         # absolute path to the colmap binary that passed the check
ENV_PREFIX   = '/content/colmap-env'      # micromamba env prefix (rung 1)
INSTALL_DIR  = '/content/colmap-install'  # cmake install prefix  (rung 2)
MAMBA_BIN    = '/content/bin/micromamba'
TIMINGS      = {}         # wall-clock results written into the Drive JSON report
ENV          = {}         # GPU / OS facts for the report
DATASET_USED = ''
SMOKE_DIR    = '/content/smoke'  # test images live here

print('=== GPU ===')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

print('\n=== CUDA compiler (may be absent on base Colab image) ===')
!nvcc --version 2>/dev/null || echo 'nvcc absent (driver CUDA still present via libcuda)'

print('\n=== OS / CPU ===')
!lsb_release -rs
!nproc

# Capture into ENV for the JSON report
def _sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.stdout.strip(), r.returncode

gpu_line, _ = _sh('nvidia-smi --query-gpu=name,memory.total,driver_version'
                  ' --format=csv,noheader')
parts = [p.strip() for p in gpu_line.split(',')]
nvcc_ver, _ = _sh("nvcc --version 2>/dev/null | grep -o 'release [0-9.]*' | head -1")
ENV.update({
    'gpu_model':      parts[0] if len(parts) > 0 else 'unknown',
    'gpu_memory':     parts[1] if len(parts) > 1 else 'unknown',
    'driver_version': parts[2] if len(parts) > 2 else 'unknown',
    'ubuntu':         _sh('lsb_release -rs')[0] or 'unknown',
    'cpu_cores':      int(_sh('nproc')[0] or '0'),
    'nvcc_version':   nvcc_ver or 'absent',
})
print('\nCaptured for report:', ENV)

## 1. RUNG 1 -- conda-forge binary (~1-2 min)

Install [micromamba](https://mamba.readthedocs.io/en/latest/micromamba-installation.html)
as a static binary into `/content` -- no condacolab kernel-restart hack needed. Then
**single-solve a fresh env containing ONLY colmap, version-pinned to `3.11.*`**:

```
micromamba create -y -q -p <ENV_PREFIX> -c conda-forge 'colmap=3.11.*=*cuda*'
```

The `=*=*cuda*` build-string selector means *any build whose string contains "cuda"*
(the CPU-only variant has a different build string and won't match). Both the `3.11.*`
pin **and** the single-solve are load-bearing -- verified live on a Colab T4, 2026-07-05:

- **Don't use the bare `colmap=*=*cuda*`.** It resolves to `colmap 4.1.0 cuda_129`, whose
  conda-forge run-deps are broken (the binary links `libfaiss` / `libOpenImageIO` but the
  package declares neither -> exit 127 at load) and which *renamed* the
  `--SiftExtraction.use_gpu` CLI option our 3.x pipeline calls rely on. `3.11.1` predates
  both problems.
- **Don't pre-pin `python=3.10` then add colmap.** That path also pulls a broken library
  set. One fresh solve of *only* colmap gives a clean dependency closure.

Expected resolve: `colmap 3.11.1 cuda_126h825ca31_105` + `ceres-solver 2.2.0`. We then
**gate on `ldd`** (fail the rung on any unresolved library) before the headless banner
check. If the create fails or the binary crashes, rung 2 builds from source. Why bother:
measured **33.3x** GPU-vs-CPU extract+match speedup on a T4 (GPU 1.5 s vs CPU 49.9 s).

In [ ]:
import os, subprocess, time

# Use globals().get so this cell is safe to re-run before cell 0 in an emergency.
MAMBA_BIN  = globals().get('MAMBA_BIN',  '/content/bin/micromamba')
ENV_PREFIX = globals().get('ENV_PREFIX', '/content/colmap-env')
TIMINGS    = globals().get('TIMINGS',    {})
ENV        = globals().get('ENV',        {})
RUNG_USED  = globals().get('RUNG_USED',  0)
COLMAP_BIN = globals().get('COLMAP_BIN', '')

def _run(cmd):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True)

# 1a: micromamba static binary -- download once, no condacolab tricks
if not os.path.isfile(MAMBA_BIN):
    print('Downloading micromamba static binary (~10 MB)...')
    t0 = time.time()
    # -C BEFORE the member name: GNU tar applies -C only to members listed after
    # it, so the other order extracts into CWD and works on Colab only by
    # accident (CWD happens to be /content).
    r = _run(
        'curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest'
        ' | tar -xvj -C /content bin/micromamba'
    )
    if r.returncode != 0 or not os.path.isfile(MAMBA_BIN):
        print('FAIL: micromamba download/extract failed')
        print(r.stderr[:1200])
        raise RuntimeError('micromamba unavailable -- cannot try rung 1')
    os.chmod(MAMBA_BIN, 0o755)
    print(f'  done ({time.time()-t0:.0f}s)')
else:
    print('micromamba already present -- skipping download')

# 1b: SINGLE-SOLVE a fresh env with ONLY colmap, version-pinned to 3.11.*.
# This one `create` replaces the old "python=3.10 env, then add colmap, then bolt
# on libfaiss" dance. Both the pin and the single-solve are load-bearing --
# verified live on a Colab T4, 2026-07-05:
#   * THE 4.1.0 TRAP: the bare wildcard 'colmap=*=*cuda*' resolves to
#     colmap 4.1.0 cuda_129, whose conda-forge run-dependency metadata is BROKEN
#     -- the binary links libfaiss.so / libOpenImageIO.so.3.1 /
#     libOpenImageIO_Util.so.3.1 but the package declares none of them, so it dies
#     at load (exit 127). Worse, 4.1.0 RENAMED the --SiftExtraction.use_gpu CLI
#     option ("unrecognised option"), silently breaking the pipeline's 3.x calls.
#   * THE PYTHON-PREPIN TRAP: creating the env with python=3.10 first and adding
#     colmap afterwards also produced missing-library chaos.
#   colmap 3.11.* predates both problems and resolves to
#   'colmap 3.11.1 cuda_126h825ca31_105' + 'ceres-solver 2.2.0' with a CLEAN
#   dependency closure (ldd shows zero 'not found').
# WHY this rung exists at all: measured 33.3x GPU-vs-CPU speedup on a T4
# (GPU 1.5 s vs CPU 49.9 s, 15 images) for feature extraction + exhaustive match.
_colmap_bin = f'{ENV_PREFIX}/bin/colmap'
if not os.path.isfile(_colmap_bin):
    print(f"Single-solve create: 'colmap=3.11.*=*cuda*' -> {ENV_PREFIX} ...")
    t0 = time.time()
    r = _run(
        f'{MAMBA_BIN} create -y -q -p {ENV_PREFIX} -c conda-forge'
        " 'colmap=3.11.*=*cuda*'"
    )
    TIMINGS['rung1_install_s'] = round(time.time() - t0, 1)
    if r.returncode != 0 or not os.path.isfile(_colmap_bin):
        print(f'  create failed / no CUDA 3.11 build indexed (exit {r.returncode},'
              f' {TIMINGS["rung1_install_s"]:.0f}s)')
        if r.stderr:
            print('  stderr:', r.stderr[:900])
        print('\n  Available conda-forge colmap builds:')
        r2 = _run(f'{MAMBA_BIN} search -c conda-forge colmap')
        print((r2.stdout or r2.stderr)[:2000])
        _colmap_bin = ''
    else:
        print(f'  done ({TIMINGS["rung1_install_s"]:.0f}s)')
else:
    print(f'{_colmap_bin} already present -- skipping create')
    TIMINGS.setdefault('rung1_install_s', 0)

# 1c: ldd closure GATE -- a first-class check. The 3.11.1 CUDA build has a clean
# dependency closure; if ANY 'not found' remains the binary would crash at load
# (this is exactly how the 4.1.0 trap manifests), so we fail the rung here rather
# than limp into a broken extract/match later.
if _colmap_bin and os.path.isfile(_colmap_bin):
    _lib = f'{ENV_PREFIX}/lib'
    r = _run(f'LD_LIBRARY_PATH={_lib} ldd {_colmap_bin}')
    _missing = [l.strip() for l in (r.stdout + r.stderr).splitlines()
                if 'not found' in l]
    if _missing:
        print(f'ldd GATE FAIL -- {len(_missing)} unresolved shared '
              f'{"library" if len(_missing) == 1 else "libraries"}:')
        for l in _missing:
            print('   ', l)
        print('  (a clean 3.11.1 env resolves everything; a broken closure like'
              ' this is the 4.1.0-style trap -- failing rung 1, rung 2 builds'
              ' from source)')
        _colmap_bin = ''
    else:
        print('ldd GATE OK -- dependency closure complete (zero "not found")')

# 1d: verify binary runs and check CUDA status in banner
if _colmap_bin and os.path.isfile(_colmap_bin):
    _env_vars = dict(
        os.environ,
        QT_QPA_PLATFORM='offscreen',  # conda colmap links Qt; headless VM has no display
        LD_LIBRARY_PATH=ENV_PREFIX + '/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
    )
    r = subprocess.run([_colmap_bin, '-h'], env=_env_vars,
                       capture_output=True, text=True, timeout=30)
    banner = r.stdout + r.stderr
    print('\ncolmap -h banner (first 15 lines):')
    print('\n'.join(banner.splitlines()[:15]))

    # 3.11.1 prints '... (Commit Unknown on Unknown with CUDA)'.
    _bl = banner.lower()
    _cuda_ok = 'with cuda' in _bl or 'cuda enabled' in _bl or (
        'cuda' in _bl and 'cuda disabled' not in _bl and 'no cuda' not in _bl
        and r.returncode == 0
    )
    print(f'\nCUDA in banner: {_cuda_ok}  exit={r.returncode}')

    if r.returncode == 0:
        # A successful =*=*cuda* pinned solve + a clean ldd gate already prove the
        # CUDA build; accept even if the banner wording shifts across point releases.
        RUNG_USED  = 1
        COLMAP_BIN = _colmap_bin
        ENV['colmap_version'] = next(
            (l.strip() for l in banner.splitlines() if 'colmap' in l.lower()), 'unknown'
        )
        print(f'\nRUNG 1 PASS -- colmap at {COLMAP_BIN}')
    else:
        print(f'\nRUNG 1 FAIL -- binary crashed (exit {r.returncode});'
              ' rung 2 will build from source')
        # Keep the missing-shared-library diagnostic -- still useful if a future
        # conda-forge revision reintroduces an undeclared dependency.
        if 'cannot open shared object file' in banner:
            _missing = banner.split('error while loading shared libraries:')[-1].split(':')[0].strip()
            print(f'  Missing shared library: {_missing} -- try:')
            print(f'  !{MAMBA_BIN} install -y -p {ENV_PREFIX} -c conda-forge <package providing it>')
else:
    print('\nRUNG 1 SKIP -- create/gate above failed; rung 2 will build from source')

print(f'\nRUNG_USED={RUNG_USED}  COLMAP_BIN={COLMAP_BIN!r}')

## 2. RUNG 2 -- source build (skipped if rung 1 succeeded)

Builds COLMAP from source with `-DGUI_ENABLED=OFF` (headless, no Qt/GLEW/GL) and a fat
`CMAKE_CUDA_ARCHITECTURES` list covering T4 (75), A100 (80), RTX 30xx (86), and
L4/RTX 40xx (89) so future users on non-A100 runtimes also benefit.

**Pin:** tag `3.10` -- the latest stable release at time of writing, with well-tested
CUDA arch detection and cmake 3.x support. Pinning avoids picking up an in-progress tag
that might have cmake or CUDA regressions. Expect ~20-35 min on Colab.

In [ ]:
import os, subprocess, time

RUNG_USED   = globals().get('RUNG_USED',   0)
INSTALL_DIR = globals().get('INSTALL_DIR', '/content/colmap-install')
ENV_PREFIX  = globals().get('ENV_PREFIX',  '/content/colmap-env')
MAMBA_BIN   = globals().get('MAMBA_BIN',   '/content/bin/micromamba')
TIMINGS     = globals().get('TIMINGS',     {})
ENV         = globals().get('ENV',         {})
COLMAP_BIN  = globals().get('COLMAP_BIN',  '')

if RUNG_USED == 1:
    print('RUNG 1 already succeeded -- skipping source build. Nothing to do here.')
else:
    print('Rung 1 failed/skipped -- building COLMAP from source.')

    # apt build deps per COLMAP docs (Ubuntu 22.04). Mesa GL dev headers are
    # required even with GUI_ENABLED=OFF on some cmake/colmap combos -- observed
    # 2026-07-05: FindDependencies.cmake did find_package(OpenGL) and configure
    # died with "Could NOT find OpenGL". We both install the headers AND pass
    # -DOPENGL_ENABLED=OFF below (belt and braces; unknown options are ignored).
    APT_DEPS = (
        'cmake ninja-build build-essential '
        'libboost-program-options-dev libboost-filesystem-dev '
        'libboost-graph-dev libboost-system-dev '
        'libeigen3-dev libflann-dev libfreeimage-dev '
        'libmetis-dev libgoogle-glog-dev libgflags-dev '
        'libsqlite3-dev libcgal-dev libceres-dev '
        'libgl1-mesa-dev libglu1-mesa-dev libglew-dev'
    )
    print('Installing apt build deps...')
    t0 = time.time()
    !apt-get -qq install -y {APT_DEPS} > /dev/null
    print(f'  apt done ({time.time()-t0:.0f}s)')

    # COLMAP >= 3.9 needs Ceres >= 2.1 (Manifold API); Ubuntu 22.04 apt ships
    # 2.0.0. If apt ceres is too old, pull a consistent ceres+glog+gflags+eigen
    # set from conda-forge into the rung-1 env and point CMAKE_PREFIX_PATH at it.
    _extra_cmake = []
    r = subprocess.run("dpkg -s libceres-dev | grep -oP 'Version: \\K[0-9]+\\.[0-9]+'",
                       shell=True, capture_output=True, text=True)
    _ceres_ver = (r.stdout.strip() or '0.0')
    if tuple(int(x) for x in _ceres_ver.split('.')[:2]) < (2, 1):
        print(f'apt ceres {_ceres_ver} < 2.1 (COLMAP>=3.9 needs Manifold API)'
              ' -- pulling ceres-solver from conda-forge...')
        if os.path.isfile(MAMBA_BIN):
            r = subprocess.run(
                f'{MAMBA_BIN} install -y -q -p {ENV_PREFIX} -c conda-forge ceres-solver',
                shell=True, capture_output=True, text=True)
            if r.returncode == 0:
                _extra_cmake.append(f'-DCMAKE_PREFIX_PATH={ENV_PREFIX}')
                print('  conda-forge ceres-solver installed; CMAKE_PREFIX_PATH set')
            else:
                print('  WARN: conda ceres install failed; configure may fail on Ceres:')
                print(r.stderr[:600])
        else:
            print('  WARN: micromamba unavailable; configure may fail on Ceres version')

    # Clone pinned stable tag -- never clone HEAD; in-progress tags can have regressions
    COLMAP_TAG = '3.10'
    COLMAP_SRC = '/content/colmap-src'
    if not os.path.isdir(COLMAP_SRC):
        print(f'Cloning colmap {COLMAP_TAG}...')
        !git clone --branch {COLMAP_TAG} --depth 1 https://github.com/colmap/colmap {COLMAP_SRC}
    else:
        print(f'{COLMAP_SRC} exists -- skipping clone')

    # Configure: no GUI, no GL SiftGPU (CUDA SiftGPU is the one we use),
    # CUDA on, fat arch list, Release build
    BUILD_DIR = '/content/colmap-build'
    os.makedirs(BUILD_DIR, exist_ok=True)
    print('\nConfiguring (cmake)...')
    # Use subprocess so semicolons in CUDA_ARCHITECTURES are not shell-interpreted.
    # capture_output so the error is IN the cell when configure fails -- the
    # first run of this notebook failed blind because output went nowhere.
    r = subprocess.run(
        ['cmake', f'-S{COLMAP_SRC}', f'-B{BUILD_DIR}', '-GNinja',
         '-DCMAKE_BUILD_TYPE=Release',
         '-DGUI_ENABLED=OFF',
         '-DOPENGL_ENABLED=OFF',
         '-DCUDA_ENABLED=ON',
         '-DCMAKE_CUDA_ARCHITECTURES=75;80;86;89',  # T4/A100/RTX30/L4
         f'-DCMAKE_INSTALL_PREFIX={INSTALL_DIR}'] + _extra_cmake,
        capture_output=True, text=True,
    )
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print('=== CMAKE STDERR (tail) ===')
        print(r.stderr[-2500:])
        raise RuntimeError(f'cmake configure failed (exit {r.returncode}) -- see stderr above')

    print('\nBuilding + installing (ninja, all cores)...')
    t0 = time.time()
    !ninja -C {BUILD_DIR} -j$(nproc) install
    TIMINGS['rung2_build_s'] = round(time.time() - t0, 1)
    print(f'  build+install done ({TIMINGS["rung2_build_s"]:.0f}s)')

    # Verify
    _colmap_bin = f'{INSTALL_DIR}/bin/colmap'
    if os.path.isfile(_colmap_bin):
        _env_vars = dict(
            os.environ,
            LD_LIBRARY_PATH=ENV_PREFIX + '/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
        )
        r = subprocess.run([_colmap_bin, '-h'], env=_env_vars,
                           capture_output=True, text=True, timeout=30)
        banner = r.stdout + r.stderr
        print('\ncolmap -h banner (first 15 lines):')
        print('\n'.join(banner.splitlines()[:15]))
        _bl = banner.lower()
        _cuda_ok = 'cuda enabled' in _bl or (
            'cuda' in _bl and 'cuda disabled' not in _bl and r.returncode == 0
        )
        print(f'\nCUDA in banner: {_cuda_ok}  exit={r.returncode}')
        if r.returncode == 0:
            RUNG_USED  = 2
            COLMAP_BIN = _colmap_bin
            ENV['colmap_version'] = next(
                (l.strip() for l in banner.splitlines() if 'colmap' in l.lower()),
                'unknown',
            )
            print(f'\nRUNG 2 PASS -- colmap at {COLMAP_BIN}')
        else:
            print('\nRUNG 2 FAIL -- binary exists but crashes; check output above')
    else:
        print(f'\nRUNG 2 FAIL -- {_colmap_bin} not found; check build output above')

    print(f'\nRUNG_USED={RUNG_USED}  COLMAP_BIN={COLMAP_BIN!r}')

## 3. Smoke test dataset

Primary: **gerrard-hall** from the official COLMAP demo datasets
(`https://demuc.de/colmap/datasets/`). Long-stable URL; well-distributed texture gives a
realistic SIFT feature load. We take the first 25 images (sorted) to keep extract+match
times under ~5 min even on CPU.

Fallback: if the URL is unreachable, download ~15 textured nature photos from Unsplash
(public domain). **Matching quality is then meaningless** -- only the GPU code path is
exercised; the verdict still tells you whether CUDA SIFT works headless.

In [ ]:
import os, subprocess, glob, shutil, time, urllib.request

SMOKE_DIR    = globals().get('SMOKE_DIR',    '/content/smoke')
DATASET_USED = globals().get('DATASET_USED', '')

IMG_DIR = f'{SMOKE_DIR}/images'
os.makedirs(IMG_DIR, exist_ok=True)

_existing = glob.glob(f'{IMG_DIR}/*')
if len(_existing) >= 10:
    print(f'Images already present ({len(_existing)} files) -- skipping download')
    DATASET_USED = DATASET_USED or 'gerrard-hall'
else:
    GH_URL = 'https://demuc.de/colmap/datasets/gerrard-hall.zip'
    GH_ZIP = '/content/gerrard-hall.zip'
    GH_SRC = '/content/gerrard-hall'
    _success = False

    if not os.path.isdir(GH_SRC):
        print('Downloading gerrard-hall dataset...')
        t0 = time.time()
        r = subprocess.run(
            ['wget', '-q', '--tries=2', '--timeout=60', '-O', GH_ZIP, GH_URL],
            capture_output=True, text=True,
        )
        if r.returncode == 0 and os.path.isfile(GH_ZIP):
            print(f'  download done ({time.time()-t0:.0f}s), unzipping...')
            subprocess.run(['unzip', '-q', '-o', GH_ZIP, '-d', '/content'], check=True)
            if os.path.isfile(GH_ZIP):
                os.remove(GH_ZIP)
            _success = True
        else:
            print(f'  WARN: download failed (exit {r.returncode}); trying fallback dataset')
            if os.path.isfile(GH_ZIP):
                os.remove(GH_ZIP)
    else:
        _success = True
        print(f'{GH_SRC} already extracted -- skipping download')

    if _success and os.path.isdir(GH_SRC):
        src_imgs = sorted(glob.glob(f'{GH_SRC}/images/*'))
        if not src_imgs:
            # Some gerrard-hall zips nest the images differently
            src_imgs = sorted(glob.glob(f'{GH_SRC}/**/*.jpg', recursive=True))
        src_imgs = src_imgs[:25]  # cap at 25 to keep matching fast
        for p in src_imgs:
            shutil.copy2(p, IMG_DIR)
        shutil.rmtree(GH_SRC)    # free disk
        DATASET_USED = 'gerrard-hall'
        print(f'Copied {len(src_imgs)} gerrard-hall images -> {IMG_DIR}')
    else:
        # Fallback: public-domain nature photos (SIFT has plenty to latch onto)
        print('!!! FALLBACK dataset: gerrard-hall unavailable')
        print('!!! Matching quality is MEANINGLESS -- only GPU code path is exercised')
        FALLBACK_URLS = [
            'https://images.unsplash.com/photo-1501854140801-50d01698950b?w=800',
            'https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800',
            'https://images.unsplash.com/photo-1464822759023-fed622ff2c3b?w=800',
            'https://images.unsplash.com/photo-1500534314209-a25ddb2bd429?w=800',
            'https://images.unsplash.com/photo-1470071459604-3b5ec3a7fe05?w=800',
            'https://images.unsplash.com/photo-1441974231531-c6227db76b6e?w=800',
            'https://images.unsplash.com/photo-1426604966848-d7adac402bff?w=800',
            'https://images.unsplash.com/photo-1500375592092-40eb2168fd21?w=800',
            'https://images.unsplash.com/photo-1504700610630-ac6aba3536d3?w=800',
            'https://images.unsplash.com/photo-1448375240586-882707db888b?w=800',
            'https://images.unsplash.com/photo-1445307806294-bff7f67ff225?w=800',
            'https://images.unsplash.com/photo-1418065460487-3e41a6c84dc5?w=800',
            'https://images.unsplash.com/photo-1472214103451-9374bd1c798e?w=800',
            'https://images.unsplash.com/photo-1490730141103-6cac27aaab94?w=800',
            'https://images.unsplash.com/photo-1469474968028-56623f02e42e?w=800',
        ]
        for i, url in enumerate(FALLBACK_URLS):
            dst = f'{IMG_DIR}/fallback_{i:03d}.jpg'
            if not os.path.isfile(dst):
                try:
                    urllib.request.urlretrieve(url, dst)
                    print(f'  fetched {i+1}/{len(FALLBACK_URLS)}')
                except Exception as e:
                    print(f'  skip {i}: {e}')
        DATASET_USED = 'synthetic-fallback'

_n = len(glob.glob(f'{IMG_DIR}/*'))
print(f'\nDataset: {DATASET_USED!r}  |  images: {_n}  |  path: {IMG_DIR}')
assert _n >= 5, f'Too few images ({_n}) -- check download output above'

## 4. The experiment -- GPU vs CPU SIFT

Run COLMAP `feature_extractor` + `exhaustive_matcher` twice on the same ~25 images:

- **Path A (GPU):** `--SiftExtraction.use_gpu 1` / `--SiftMatching.use_gpu 1` -- the
  **headless crash test**. Must complete with exit code 0 for PASS.
- **Path B (CPU):** same but `use_gpu 0` -- baseline timing for the speedup table.

Each path writes its database to a separate workdir so they are fully independent.
A quick `mapper` run on the GPU database at the end confirms end-to-end health.

In [ ]:
import os, subprocess, time

RUNG_USED  = globals().get('RUNG_USED',  0)
COLMAP_BIN = globals().get('COLMAP_BIN', '')
SMOKE_DIR  = globals().get('SMOKE_DIR',  '/content/smoke')
ENV_PREFIX = globals().get('ENV_PREFIX', '/content/colmap-env')
TIMINGS    = globals().get('TIMINGS',    {})

if not COLMAP_BIN or RUNG_USED == 0:
    raise RuntimeError(
        'COLMAP_BIN not set -- run cells 4 (rung 1) and/or 6 (rung 2) first'
    )

IMG_DIR = f'{SMOKE_DIR}/images'
WD_GPU  = f'{SMOKE_DIR}/workdir_gpu'
DB_GPU  = f'{WD_GPU}/database.db'
os.makedirs(WD_GPU, exist_ok=True)

# conda colmap links Qt and expects QT_QPA_PLATFORM=offscreen on headless VMs.
# Source build (rung 2) has no Qt, but setting the var is harmless.
_env = dict(
    os.environ,
    QT_QPA_PLATFORM='offscreen',
    LD_LIBRARY_PATH=ENV_PREFIX + '/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
)

def _colmap_run(args):
    """Run COLMAP, stream last 30 lines, return (elapsed_s, returncode)."""
    t0 = time.time()
    print('$', COLMAP_BIN.split('/')[-1], ' '.join(args[:3]), '...')
    r = subprocess.run([COLMAP_BIN] + args, env=_env,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for l in r.stdout.splitlines()[-30:]:
        print(l)
    if r.returncode != 0:
        print(f'  [exit {r.returncode}]')
    return round(time.time() - t0, 1), r.returncode

print('=' * 60)
print('PATH A: GPU SIFT  (the headless crash test)')
print('=' * 60)

print('\n-- feature_extractor (GPU) --')
t, rc = _colmap_run([
    'feature_extractor',
    '--database_path', DB_GPU,
    '--image_path',    IMG_DIR,
    '--SiftExtraction.use_gpu', '1',
])
TIMINGS['gpu_feat_s'] = t
print(f'\n  feature_extractor (GPU): {t:.1f}s  exit={rc}')
assert rc == 0, (
    f'feature_extractor GPU crashed (exit {rc}) -- '
    'CUDA SIFT not working headless; check log above'
)

print('\n-- exhaustive_matcher (GPU) --')
t, rc = _colmap_run([
    'exhaustive_matcher',
    '--database_path', DB_GPU,
    '--SiftMatching.use_gpu', '1',
])
TIMINGS['gpu_match_s'] = t
TIMINGS['gpu_total_s'] = TIMINGS['gpu_feat_s'] + t
print(f'\n  exhaustive_matcher (GPU): {t:.1f}s  exit={rc}')
assert rc == 0, (
    f'exhaustive_matcher GPU crashed (exit {rc}) -- '
    'CUDA SIFT matching broken; check log above'
)

print(f'\nGPU path total: {TIMINGS["gpu_total_s"]:.1f}s')

In [ ]:
import os, subprocess, time

COLMAP_BIN = globals().get('COLMAP_BIN', '')
SMOKE_DIR  = globals().get('SMOKE_DIR',  '/content/smoke')
ENV_PREFIX = globals().get('ENV_PREFIX', '/content/colmap-env')
TIMINGS    = globals().get('TIMINGS',    {})

if not COLMAP_BIN:
    raise RuntimeError('COLMAP_BIN not set -- run cells 4/6 first')

IMG_DIR = f'{SMOKE_DIR}/images'
WD_CPU  = f'{SMOKE_DIR}/workdir_cpu'
DB_CPU  = f'{WD_CPU}/database.db'
os.makedirs(WD_CPU, exist_ok=True)

_env = dict(
    os.environ,
    QT_QPA_PLATFORM='offscreen',
    LD_LIBRARY_PATH=ENV_PREFIX + '/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
)

def _colmap_run(args):
    t0 = time.time()
    print('$', COLMAP_BIN.split('/')[-1], ' '.join(args[:3]), '...')
    r = subprocess.run([COLMAP_BIN] + args, env=_env,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for l in r.stdout.splitlines()[-30:]:
        print(l)
    if r.returncode != 0:
        print(f'  [exit {r.returncode}]')
    return round(time.time() - t0, 1), r.returncode

print('=' * 60)
print('PATH B: CPU SIFT  (baseline timing)')
print('=' * 60)

print('\n-- feature_extractor (CPU) --')
t, rc = _colmap_run([
    'feature_extractor',
    '--database_path', DB_CPU,
    '--image_path',    IMG_DIR,
    '--SiftExtraction.use_gpu', '0',
])
TIMINGS['cpu_feat_s'] = t
print(f'\n  feature_extractor (CPU): {t:.1f}s  exit={rc}')
if rc != 0:
    print(f'WARN: CPU feature_extractor failed (exit {rc}); CPU timing is invalid')

print('\n-- exhaustive_matcher (CPU) --')
t, rc = _colmap_run([
    'exhaustive_matcher',
    '--database_path', DB_CPU,
    '--SiftMatching.use_gpu', '0',
])
TIMINGS['cpu_match_s'] = t
TIMINGS['cpu_total_s'] = TIMINGS.get('cpu_feat_s', 0) + t
print(f'\n  exhaustive_matcher (CPU): {t:.1f}s  exit={rc}')

_gpu_t = TIMINGS.get('gpu_total_s', 0)
_cpu_t = TIMINGS.get('cpu_total_s', 1)
if _gpu_t > 0 and _cpu_t > 0:
    TIMINGS['speedup_x'] = round(_cpu_t / _gpu_t, 1)
    print(f'\nSpeedup: {_cpu_t:.1f}s / {_gpu_t:.1f}s = {TIMINGS["speedup_x"]}x')
print('\nTIMINGS:', TIMINGS)

In [ ]:
import os, subprocess, sqlite3, time

COLMAP_BIN = globals().get('COLMAP_BIN', '')
SMOKE_DIR  = globals().get('SMOKE_DIR',  '/content/smoke')
ENV_PREFIX = globals().get('ENV_PREFIX', '/content/colmap-env')
TIMINGS    = globals().get('TIMINGS',    {})

if not COLMAP_BIN:
    raise RuntimeError('COLMAP_BIN not set -- run cells 4/6 first')

IMG_DIR    = f'{SMOKE_DIR}/images'
DB_GPU     = f'{SMOKE_DIR}/workdir_gpu/database.db'
DB_CPU     = f'{SMOKE_DIR}/workdir_cpu/database.db'

# -- DB sanity check: row counts confirm features and matches were written --
# keypoints table has one row per image (the keypoint array is stored as a blob);
# row count = number of images successfully processed.
# matches table has one row per image pair with overlapping keypoints.
KP, MATCHES = {}, {}
for label, db_path in [('gpu', DB_GPU), ('cpu', DB_CPU)]:
    if not os.path.isfile(db_path):
        print(f'WARN: {label} database missing at {db_path}')
        KP[label] = MATCHES[label] = 0
        continue
    try:
        conn = sqlite3.connect(db_path)
        KP[label]      = conn.execute('SELECT COUNT(*) FROM keypoints').fetchone()[0]
        MATCHES[label] = conn.execute('SELECT COUNT(*) FROM matches').fetchone()[0]
        conn.close()
    except Exception as e:
        print(f'WARN: sqlite query failed on {label}: {e}')
        KP[label] = MATCHES[label] = -1

print('Database row counts (keypoints = #images processed; matches = #image pairs):')
print(f'  {"path":6s}  {"kp_rows":>10s}  {"match_rows":>12s}')
for label in ('gpu', 'cpu'):
    print(f'  {label:6s}  {KP[label]:>10d}  {MATCHES[label]:>12d}')

TIMINGS['gpu_kp_rows']    = KP.get('gpu', 0)
TIMINGS['gpu_match_rows'] = MATCHES.get('gpu', 0)
TIMINGS['cpu_kp_rows']    = KP.get('cpu', 0)
TIMINGS['cpu_match_rows'] = MATCHES.get('cpu', 0)

# -- Quick mapper sanity on the GPU database --
# A successful reconstruction proves COLMAP can go end-to-end, not just extract/match.
SPARSE_DIR = f'{SMOKE_DIR}/workdir_gpu/sparse'
os.makedirs(SPARSE_DIR, exist_ok=True)
_env = dict(
    os.environ,
    QT_QPA_PLATFORM='offscreen',
    LD_LIBRARY_PATH=ENV_PREFIX + '/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
)
if not os.listdir(SPARSE_DIR):
    print('\nRunning mapper on GPU database (quick sanity)...')
    t0 = time.time()
    r = subprocess.run(
        [COLMAP_BIN, 'mapper',
         '--database_path', DB_GPU,
         '--image_path',    IMG_DIR,
         '--output_path',   SPARSE_DIR],
        env=_env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    TIMINGS['mapper_s'] = round(time.time() - t0, 1)
    for l in r.stdout.splitlines()[-40:]:
        print(l)
    print(f'  mapper: {TIMINGS["mapper_s"]:.1f}s  exit={r.returncode}')
    models = sorted(d for d in os.listdir(SPARSE_DIR)
                    if os.path.isdir(f'{SPARSE_DIR}/{d}'))
    print(f'  sparse models: {len(models)}')
    for m in models:
        imgs_file = f'{SPARSE_DIR}/{m}/images.txt'
        if os.path.isfile(imgs_file):
            with open(imgs_file) as f:
                reg = sum(1 for l in f if l.strip() and not l.startswith('#')) // 2
            print(f'    model {m}: {reg} registered images')
else:
    print(f'Sparse dir already populated -- skipping mapper')

## 5. Persist artifact + report to Drive

Tar the working COLMAP artifact (rung 1: micromamba env prefix; rung 2: cmake install
dir) to `MyDrive/4dgs/colmap-cuda/` and write `colmap_cuda_report.json` alongside it.
A future `bootstrap.sh --colmap-cuda` can pull the tarball from Drive and extract it
instead of rebuilding from scratch every run.

Drive is mounted here (last, matching the other notebooks' pattern) so the earlier
experiment cells run without needing Drive access.

In [ ]:
import os, subprocess, json, time

RUNG_USED    = globals().get('RUNG_USED',    0)
COLMAP_BIN   = globals().get('COLMAP_BIN',   '')
ENV_PREFIX   = globals().get('ENV_PREFIX',   '/content/colmap-env')
INSTALL_DIR  = globals().get('INSTALL_DIR',  '/content/colmap-install')
TIMINGS      = globals().get('TIMINGS',      {})
ENV          = globals().get('ENV',          {})
DATASET_USED = globals().get('DATASET_USED', 'unknown')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/4dgs/colmap-cuda'
os.makedirs(DRIVE_OUT, exist_ok=True)

# Choose what to tar based on which rung succeeded
if RUNG_USED == 1:
    TAR_SRC  = ENV_PREFIX
    TAR_NAME = 'colmap-env.tar.gz'
elif RUNG_USED == 2:
    TAR_SRC  = INSTALL_DIR
    TAR_NAME = 'colmap-install.tar.gz'
else:
    TAR_SRC = TAR_NAME = None
    print('WARN: no rung succeeded; nothing to tar')

if TAR_SRC and os.path.isdir(TAR_SRC):
    tar_dst = f'{DRIVE_OUT}/{TAR_NAME}'
    if not os.path.isfile(tar_dst):
        print(f'Tarring {TAR_SRC} -> {tar_dst} ...')
        t0 = time.time()
        r = subprocess.run(
            ['tar', '-czf', tar_dst, '-C', os.path.dirname(TAR_SRC),
             os.path.basename(TAR_SRC)],
            capture_output=True, text=True,
        )
        if r.returncode == 0:
            size_mb = os.path.getsize(tar_dst) / 1e6
            print(f'  done ({time.time()-t0:.0f}s, {size_mb:.0f} MB) -> {tar_dst}')
        else:
            print(f'  tar failed (exit {r.returncode}):', r.stderr[:400])
    else:
        print(f'{tar_dst} already exists -- skipping tar')

# Build report dict
_gpu_t = TIMINGS.get('gpu_total_s', 0)
_cpu_t = TIMINGS.get('cpu_total_s', 0)
report = {
    'rung_used':         RUNG_USED,
    'gpu_model':         ENV.get('gpu_model',      'unknown'),
    'driver_version':    ENV.get('driver_version', 'unknown'),
    'nvcc_version':      ENV.get('nvcc_version',   'unknown'),
    'ubuntu':            ENV.get('ubuntu',          'unknown'),
    'cpu_cores':         ENV.get('cpu_cores',       0),
    'colmap_version':    ENV.get('colmap_version',  'unknown'),
    'rung1_install_s':   TIMINGS.get('rung1_install_s', None),
    'rung2_build_s':     TIMINGS.get('rung2_build_s',   None),
    'gpu_feat_s':        TIMINGS.get('gpu_feat_s',  None),
    'gpu_match_s':       TIMINGS.get('gpu_match_s', None),
    'gpu_total_s':       _gpu_t or None,
    'cpu_feat_s':        TIMINGS.get('cpu_feat_s',  None),
    'cpu_match_s':       TIMINGS.get('cpu_match_s', None),
    'cpu_total_s':       _cpu_t or None,
    'speedup_x':         TIMINGS.get('speedup_x',   None),
    'dataset':           DATASET_USED,
    'gpu_kp_rows':       TIMINGS.get('gpu_kp_rows',    0),
    'gpu_match_rows':    TIMINGS.get('gpu_match_rows',  0),
    'cpu_kp_rows':       TIMINGS.get('cpu_kp_rows',    0),
    'cpu_match_rows':    TIMINGS.get('cpu_match_rows',  0),
    'mapper_s':          TIMINGS.get('mapper_s',    None),
}
report_path = f'{DRIVE_OUT}/colmap_cuda_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f'\nReport -> {report_path}')
print(json.dumps(report, indent=2))

## 6. Verdict

In [ ]:
RUNG_USED    = globals().get('RUNG_USED',    0)
COLMAP_BIN   = globals().get('COLMAP_BIN',   '')
TIMINGS      = globals().get('TIMINGS',      {})
ENV          = globals().get('ENV',          {})
DATASET_USED = globals().get('DATASET_USED', 'unknown')
DRIVE_OUT    = '/content/drive/MyDrive/4dgs/colmap-cuda'

_gpu_t  = TIMINGS.get('gpu_total_s', 0)
_cpu_t  = TIMINGS.get('cpu_total_s', 0)
_spd    = TIMINGS.get('speedup_x',   None)
_gpu_kp = TIMINGS.get('gpu_kp_rows', 0)
_gpu_m  = TIMINGS.get('gpu_match_rows', 0)

# PASS = CUDA SIFT ran headless and produced features + matches.
# On the synthetic fallback dataset the photos share no scene, so ZERO verified
# matches is expected -- require matches only on a real multi-view dataset.
_need_matches = DATASET_USED != 'synthetic-fallback'
PASS = (RUNG_USED > 0 and bool(COLMAP_BIN) and _gpu_kp > 0
        and (_gpu_m > 0 or not _need_matches))

print('=' * 60)
if PASS:
    print('PASS -- CUDA COLMAP works headless on this runtime')
    print('=' * 60)
    print(f'  Rung        : {RUNG_USED} ({"conda-forge binary" if RUNG_USED==1 else "source build"})')
    print(f'  Binary      : {COLMAP_BIN}')
    print(f'  GPU         : {ENV.get("gpu_model","?")}')
    print(f'  COLMAP      : {ENV.get("colmap_version","?")}')
    print(f'  Dataset     : {DATASET_USED}')
    if not _need_matches:
        print('  NOTE: fallback dataset -- match counts are meaningless by design;')
        print('        PASS is based on headless GPU extraction only')
    print()
    print('  Timing (extract + match):')
    print(f'    GPU  : {_gpu_t:.1f}s')
    print(f'    CPU  : {_cpu_t:.1f}s')
    if _spd is not None:
        print(f'    Speedup: {_spd}x')
    print()
    print('  DB health:')
    print(f'    GPU  kp_rows={_gpu_kp}  match_rows={_gpu_m}')
    print(f'    CPU  kp_rows={TIMINGS.get("cpu_kp_rows",0)}')
    print()
    print(f'  Drive artifact : {DRIVE_OUT}/')
    print(f'  Drive report   : {DRIVE_OUT}/colmap_cuda_report.json')
    print()
    print('  Next step: wire COLMAP_BIN into bootstrap.sh --colmap-cuda')
    print('  (extract tar from Drive, prepend to PATH or symlink into /usr/local/bin)')
else:
    print('FAIL -- CUDA COLMAP did not complete')
    print('=' * 60)
    print(f'  RUNG_USED={RUNG_USED}  COLMAP_BIN={COLMAP_BIN!r}')
    print(f'  gpu_kp_rows={_gpu_kp}  gpu_match_rows={_gpu_m}')
    print()
    if RUNG_USED == 0:
        print('  Both rungs failed to install/build COLMAP.')
        print('  Rung 1: check micromamba download (try curl URL manually).')
        print('  Rung 2: scroll to cmake output above for config errors')
        print('          (missing CUDA toolkit, missing apt deps, etc.).')
    elif _gpu_kp == 0:
        print('  COLMAP installed but feature extraction produced no keypoints.')
        print('  CUDA SIFT likely crashed silently. Check cell 10 output.')
        print('  Try: re-run cell 10 with --SiftExtraction.num_threads 1')
    else:
        print('  Features extracted but matching failed.')
        print('  Check cell 10 output. Try: --SiftMatching.num_threads 1')
print('=' * 60)